In [1]:
import numpy as np
import torch
import os
import sys
import json
import jsonpickle as jpickle

current_dir = os.path.abspath('')
os.chdir(current_dir)
sys.path.append(os.path.join(current_dir,'code','BalancingControl'))

import two_stage_utils as tu
import inference_utils as iu
import inference as inf

torch threads 1


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on device cpu
torch threads 1


In [2]:
available_datasets_list = ["Chen_et_al_data", "FeherDaSilva_data/magic_carpet", "Kool_et_al_data/daw_paradigm"]

possible_models_list = ["BCC_2pars_planning", "BCC_4pars_planning_repetition_weight", 
                "BCC_4pars_planning_cached", "BCC_6pars_planning_repetition_weight_cached",
                "MFMB_4pars_mf_mb_Orig", "MFMB_6pars_mf_mb_Orig_prior"]

In [3]:
def load_measure(dataset, model, measure_name):

    measure_file = os.path.join(dataset, "results", "inference", model+"_inference", model+"_inference__"+measure_name+".json")

    with open(measure_file, 'r') as infile:
        pickled_measure = json.load(infile)
    measure = jpickle.decode(pickled_measure)

    return measure

In [4]:
measure_list = []
type_list = []
model_list = []
dataset_list = []

for dataset in available_datasets_list:

    for model in possible_models_list:

        # load WAIC
        WAIC = load_measure(dataset, model, "WAIC")
        measure_list.append(WAIC)
        type_list.append("WAIC")
        model_list.append(model)
        dataset_list.append(dataset)

        # load log likelihood
        ll = load_measure(dataset, model, "log_likelihood")
        measure_list.append(ll)
        type_list.append("log_likelihood")
        model_list.append(model)
        dataset_list.append(dataset)

In [5]:
WAIC_model_list = []
ll_model_list = []
type_list = []
model_list = []


for model in possible_models_list:
    WAIC_list = []
    ll_list = []

    for dataset in available_datasets_list:

        # load WAIC
        WAIC = load_measure(dataset, model, "WAIC")
        WAIC_list.append(WAIC)

        # load log likelihood
        ll = load_measure(dataset, model, "log_likelihood")
        ll_list.append(ll)

    WAIC_tensor = torch.cat(WAIC_list)
    ll_tensor = torch.cat(ll_list)

    WAIC_model_list.append(WAIC_tensor)
    ll_model_list.append(ll_tensor)
    model_list.append(model)

In [8]:
all_WAICs = torch.stack(WAIC_model_list, dim=-1)
# print(all_WAICs.shape)
# print(all_WAICs.argmin())
# print(all_WAICs)

iu.calculate_exceedance_prob(-0.5*all_WAICs[...,:4])

p model mean according to measure tensor([0.3186, 0.2861, 0.0902, 0.3051])
best model: tensor(0) exceedance prob tensor(0.5680)
is significantly different from uniform? TtestResult(statistic=np.float64(67.49755214229013), pvalue=np.float64(2.3631851694775102e-253), df=np.int64(499))


In [12]:
BCC6_MFMB6_WAICs = torch.stack([WAIC_model_list[3],WAIC_model_list[5]], dim=-1)
# print(all_WAICs.shape)
# print(all_WAICs.argmin())
# print(all_WAICs)

iu.calculate_exceedance_prob(-0.5*BCC6_MFMB6_WAICs)

p model mean according to measure tensor([0.2008, 0.7992])
best model: tensor(1) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(338.58973086814467), pvalue=np.float64(0.0), df=np.int64(499))
